<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# DataWizard: Análisis de datos impulsado por IA

## El desafío

En el mundo actual, impulsado por los datos, la información valiosa se encuentra oculta en hojas de cálculo y conjuntos de datos que la mayoría de los profesionales no tienen las habilidades técnicas para analizar. Los dueños de negocios, gerentes y expertos en el sector saben que sus datos son valiosos, pero carecen de los conocimientos de programación necesarios para extraer información relevante.

## Acerca de este laboratorio

En este laboratorio, crearás un agente con inteligencia artificial que ayudará a usuarios sin conocimientos técnicos a realizar tareas de ciencia de datos mediante lenguaje natural. Aprenderás a:

1. Crear una colección de herramientas LangChain que realicen tareas básicas de ciencia de datos:

    - Listar conjuntos de datos disponibles

    - Cargar y analizar archivos CSV

    - Generar resúmenes y estadísticas de conjuntos de datos

    - Entrenar y evaluar modelos de aprendizaje automático

2. Documentar claramente el propósito y el formato de salida de cada herramienta para garantizar que el agente de IA pueda utilizarlas eficazmente.

3. Demostrar por qué los agentes conversacionales convencionales tienen limitaciones al trabajar con datos estructurados.

4. Implementar un agente ejecutor que pueda gestionar un flujo de trabajo de varios pasos para el análisis de datos.

Al finalizar este laboratorio, comprenderás cómo combinar modelos de lenguaje con herramientas especializadas para crear aplicaciones prácticas que hagan que la ciencia de datos sea accesible para todos.

## __Table of Contents__

<ol>
    <li>
        <a href="#Agents">Agents</a>
        <ol>
            <li><a href="#Agent-creation-and-limitations">Agent creation and limitations</a></li>
            <li><a href="#Agent-executor-ReAct">Agent executor ReAct</a></li>
        </ol>
    </li>
</ol>

<li><a href="#Authors">Authors</a></li>
<li><a href="#Contributors">Contributors</a></li>


## 1. Objetivos

Al finalizar este proyecto, podrás:
1. **Crear herramientas especializadas para LangChain**: Desarrollar herramientas personalizadas para realizar tareas clave de ciencia de datos, como el descubrimiento, análisis y modelado de conjuntos de datos.

2. **Implementar mecanismos de almacenamiento en caché eficientes**: Crear un sistema que almacene y gestione conjuntos de datos en memoria para optimizar el rendimiento en múltiples consultas.

3. **Diseñar una interfaz de lenguaje natural**: Conectar un modelo de lenguaje a tus herramientas, permitiendo el acceso conversacional a las capacidades de ciencia de datos.

4. **Desarrollar agentes sensibles al contexto**: Crear un agente ejecutor capaz de mantener el estado y ejecutar flujos de trabajo de análisis de datos de varios pasos.

5. **Gestionar errores de forma eficaz**: Implementar un manejo de errores robusto para operaciones de archivos y tareas de procesamiento de datos.

6. **Probar tu asistente con consultas reales**: Evaluar el sistema con preguntas de negocio prácticas que demuestren su capacidad para hacer que la ciencia de datos sea accesible.

Este proyecto te proporciona las habilidades necesarias para cerrar la brecha entre los usuarios no técnicos y el análisis de datos, democratizando el acceso a la analítica avanzada a través de la conversación.

## 2. Configuración


Para este laboratorio, utilizarás las siguientes bibliotecas:

* [`langchain`](https://python.langchain.com/docs/get_started/introduction) para crear aplicaciones de IA modulares con herramientas y agentes.

* [`langchain-openai`](https://python.langchain.com/docs/integrations/llms/openai) para conectar LangChain con los modelos de lenguaje de OpenAI.

* [`openai`](https://github.com/openai/openai-python) para acceder a los modelos de IA que impulsan tu interfaz conversacional.

* [`pandas`](https://pandas.pydata.org/) para la manipulación y el análisis de conjuntos de datos CSV.

* [`numpy`](https://numpy.org/) para operaciones numéricas y procesamiento de matrices.
* [`scikit-learn`](https://scikit-learn.org/stable/) para implementar modelos de aprendizaje automático y métricas de evaluación.

* [`matplotlib`](https://matplotlib.org/) para crear visualizaciones de datos basadas en los resultados del análisis.

* [`seaborn`](https://seaborn.pydata.org/) para visualizaciones estadísticas mejoradas de patrones en conjuntos de datos.

### *2.1. Instalación de las Bibliotecas Necesarias*

Las siguientes bibliotecas necesarias no están preinstaladas en el su entorno. Debe ejecutar la siguiente celda para instalarlas. Este paso puede tardar varios minutos, tenga paciencia.

```bash
pip install -r requirements.txt
```

### *2.2. Importación de las Bibliotecas Necesarias*

Importa aquí todas las bibliotecas necesarias:

In [1]:
import glob
import numpy as np
import os
import pandas as pd
import sklearn
from typing import List, Optional

### *2.3. Aviso Legal Sobre la API*
Este laboratorio utiliza modelos de lenguaje de programación (LLM) proporcionados por OpenAI. Este entorno se ha configurado para permitir el uso de LLM sin claves API, de modo que pueda solicitarlos **gratis (con limitaciones)**. Por lo tanto, si desea ejecutar este cuaderno **localmente** fuera del entorno JupyterLab de Skills Network, deberá configurar sus propias claves API. Tenga en cuenta que el uso de sus propias claves API implicará cargos personales.

### *2.4. Ejecución Local*

Si ejecuta este laboratorio localmente, deberá configurar su propia clave API. Este laboratorio utiliza la función `init_chat_model` de `langchain`. Para usar el modelo, debe establecer la variable de entorno `OPENAI_API_KEY` con su clave API de OpenAI. **NO** ejecute la celda siguiente si no está ejecutando el laboratorio localmente, ya que provocará errores.

In [2]:
# os.environ["OPENAI_API_KEY"] = "Aqui tienes tu clave API de OpenAI"

## 3. Herramientas de LangChain

Las herramientas de LangChain son interfaces que permiten a un modelo de IA (como GPT-4) interactuar con sistemas externos, recuperar datos o realizar acciones más allá de la simple generación de texto. Estas herramientas funcionan como API o llamadas a funciones que la IA puede invocar cuando sea necesario.

Primero, crearás una herramienta para identificar los conjuntos de datos disponibles en el directorio local. Esta herramienta ayuda a tu agente a descubrir qué archivos CSV están disponibles para su análisis sin que el usuario tenga que especificar explícitamente los nombres de archivo. Se asume que los archivos CSV tienen nombres descriptivos que indican su contenido. Este suele ser el primer paso en tu flujo de trabajo de análisis de datos: descubrir qué datos están disponibles antes de cargarlos o analizarlos.

Las herramientas de LangChain son funciones sencillas de Python envueltas con el decorador `@tool` (importadas desde `langchain_core.tools`). Permiten a los Modelos de Lenguaje a Gran Escala (LLM) llamar a funciones específicas, lo que posibilita flujos de trabajo estructurados e integraciones con herramientas externas.

```python
from langchain_core.tools import tool

@tool
def my_tool(input: <type>) -> <output_type>:
    """
    Breve descripción de la función de la herramienta.

    Argumentos:
    entrada (<tipo>): Explicación del argumento de entrada.
    Valor de retorno:
    <tipo_de_salida>: Explicación del valor devuelto.
    """

    # Implementa tu lógica aquí.
    return tool_output
```

Componentes de la herramienta:

- **Decorador `@tool`:** Marca la función como una herramienta, permitiendo que LangChain la integre y la exponga al LLM.

- **Argumentos de entrada:** Los parámetros que acepta la función de la herramienta, junto con anotaciones de tipo para mayor claridad.

- **Descripción de la herramienta:** Una explicación clara y concisa que LangChain y el LLM utilizan para comprender cuándo y cómo llamar a la herramienta.

- **Tipo de retorno:** Especifica el tipo de datos que devolverá la herramienta, mejorando la claridad y la fiabilidad.

    - **`.name`:** Derivado automáticamente del nombre de la función de Python; utilizado por LangChain para identificar la herramienta.

    - **`.description`:** Extraído automáticamente de la cadena de documentación de la función; ayuda al LLM a comprender el propósito de la herramienta.

    - **`.args`:** Representa los argumentos de entrada con sus tipos asociados, lo que permite a LangChain validar y pasar los valores correctos a la función de la herramienta.

Vamos a crear la primera herramienta de LangChain, que lista todos los archivos CSV en el directorio actual.

- `os.getcwd()` recupera el directorio de trabajo actual.

- `os.path.join(os.getcwd(), "*.csv")` construye un patrón de ruta que coincide con todos los archivos CSV (`*` coincide con todos los nombres de archivo que terminan en `.csv`).

- `glob.glob(pattern)` devuelve una lista de archivos que coinciden con el patrón dado.


In [3]:
from langchain_core.tools import tool

@tool
def listar_archivos_csv() -> Optional[List[str]]:
    """Lista todos los nombres de archivos CSV en el directorio local.
    Devuelve:
    - Una lista con los nombres de los archivos CSV.
    - Si no se encuentran archivos CSV, devuelve None.
    """

    csv_files = glob.glob(os.path.join(os.getcwd(), "data/*.csv"))

    if not csv_files:
        return None
    
    return ['data/' + os.path.basename(file) for file in csv_files]

Puedes imprimir los atributos útiles de la herramienta, lo que ayuda durante la depuración y permite a LangChain identificar claramente el propósito y las entradas de cada función:

In [4]:
print("Tool Name:", listar_archivos_csv.name)
print("Tool Description:", listar_archivos_csv.description)
print("Tool Arguments:", listar_archivos_csv.args)

Tool Name: listar_archivos_csv
Tool Description: Lista todos los nombres de archivos CSV en el directorio local.
    Devuelve:
    - Una lista con los nombres de los archivos CSV.
    - Si no se encuentran archivos CSV, devuelve None.
Tool Arguments: {}


### *3.1. Herramienta de Almacenamiento en Caché de Conjuntos de Datos*

A medida que desarrollas herramientas más complejas, necesitas gestionar eficientemente los conjuntos de datos en memoria. Dado que los modelos de lenguaje se comunican mediante texto, enviar conjuntos de datos completos en cada respuesta desperdiciaría tokens y espacio en la ventana de contexto. Para solucionar esto, crearás una caché global que almacena los DataFrames después de su primera carga. Este enfoque ofrece varias ventajas:

  1. Reduce el uso de tokens al referenciar los conjuntos de datos por nombre en lugar de por contenido.

  2. Mejora el rendimiento al cargar los datos solo una vez.

  3. Mantiene la disponibilidad de los conjuntos de datos entre diferentes llamadas a la herramienta.

La siguiente herramienta permite al agente precargar conjuntos de datos en este sistema de caché.


In [5]:
DATAFRAME_CACHE = {}

@tool
def precarga_datasets(rutas: List[str]) -> str:
    """
   Carga archivos CSV en una caché global si aún no están cargados.

    Esta función ayuda a gestionar eficientemente los conjuntos de datos cargándolos una sola vez
    y almacenándolos en memoria para su uso posterior. Sin el almacenamiento en caché,
    se desperdiciarían tokens al describir repetidamente el contenido de los conjuntos de datos en las respuestas del agente.

    Argumentos:
        rutas: Una lista de rutas de archivo a los archivos CSV.

    Devuelve:
        Un mensaje que resume qué conjuntos de datos se cargaron o ya estaban en caché.
    """
    
    cargados = []
    almacenados_cache = []
    for path in rutas:
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] = pd.read_csv(path)
            cargados.append(path)
        else:
            almacenados_cache.append(path)
    
    return (
        f"Cargados datasets: {cargados}\n"
        f"Ya almacenados en cache: {almacenados_cache}"
    )

Podría pensarse que la palabra clave `global` funcionaría mejor que `DATAFRAME CACHE`, pero al usar funciones en Python con agentes LangChain, simplemente escribir la palabra clave `global` no basta para mantener los datos entre diferentes llamadas a herramientas. Esto se debe a que cada vez que el agente ejecuta una herramienta, podría hacerlo en un entorno de ejecución independiente, lo que provocaría que las variables `global` se reiniciaran. En cambio, usar un diccionario a nivel de módulo (`DATAFRAME_CACHE`) que reside fuera de cualquier función crea un espacio de almacenamiento persistente al que todas las herramientas pueden acceder sin necesidad de pasarlo explícitamente. Este enfoque funciona de forma fiable en múltiples llamadas a funciones, incluso cuando ocurren en diferentes contextos, y mantiene las interfaces de las funciones limpias al evitar la necesidad de pasar la caché como un parámetro adicional a cada herramienta.

Las cadenas de documentación bien estructuradas son esenciales para las herramientas basadas en LLM, ya que sirven como instrucciones para el agente de IA. El agente se basa en estas descripciones para comprender cuándo y cómo usar cada herramienta. Del mismo modo, las salidas formateadas ayudan al agente a analizar e interpretar los resultados correctamente. Asegúrese siempre de que las funciones de su herramienta tengan cadenas de documentación claras y devuelvan resultados bien estructurados que sean fáciles de entender tanto para los humanos como para la IA.

### *3.2. Herramienta de Resumen*
A continuación, cree una herramienta para proporcionar resúmenes de conjuntos de datos con información estadística clave. Esta herramienta ofrece al agente una visión general rápida de cada conjunto de datos sin transferir todo el contenido. Examina la estructura de cada archivo CSV y devuelve metadatos que ayudan al agente a comprender con qué tipo de datos está trabajando.

Las anotaciones de tipo son extremadamente importantes en las herramientas por varias razones:

1. Proporcionan documentación clara de las entradas y salidas esperadas.

2. Permiten la verificación estática de tipos para detectar errores antes de la ejecución.

3. Y lo que es más importante, ayudan al agente de IA a comprender exactamente qué datos proporcionar y qué formato esperar como respuesta.

Para esta herramienta, las anotaciones de tipo anidadas (List[Dict[str, Any]]) definen con precisión los datos estructurados que se devolverán. Esto permite al agente analizar y utilizar los resultados correctamente en los pasos de razonamiento posteriores.

In [6]:
from typing import List, Optional,Dict,Any

@tool
def get_resumenes_dataset(rutas_dataset: List[str]) -> List[Dict[str, Any]]:
    """
    Analiza varios archivos CSV y devuelve resúmenes de metadatos para cada uno.

    Argumentos:
        rutas_dataset (List[str]): Una lista de rutas de archivo a conjuntos de datos CSV.

    Devuelve:
        List[Dict[str, Any]]: Una lista de resúmenes, uno por conjunto de datos, cada uno con:
        - "file_name": La ruta del archivo del conjunto de datos.
        - "column_names": Una lista de nombres de columnas en el conjunto de datos.
        - "data_types": Un diccionario que relaciona los nombres de las columnas con sus tipos de datos (como cadenas).
    """

    summaries = []

    for path in rutas_dataset:
        # Cargar y almacenar en caché el conjunto de datos si aún no está almacenado en caché.
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] = pd.read_csv(path)
        
        df = DATAFRAME_CACHE[path]

        # Construir resumenes.
        summary = {
            "nombre_archivo": path,
            "nombres_columnas": df.columns.tolist(),
            "tipo_datos": df.dtypes.astype(str).to_dict()
        }

        summaries.append(summary)

    return summaries

### *3.3. Herramienta de Ejecución de Métodos de DataFrame*

Ahora que comprende los conceptos básicos de sus conjuntos de datos, necesita una forma flexible de explorarlos y analizarlos. Al igual que los científicos de datos utilizan diversos métodos de pandas DataFrame (`head()`, `describe()`, `info()`, etc.), su agente necesita la capacidad de aplicar estos métodos a sus conjuntos de datos en caché.

Utilizará la función `getattr()` de Python, que le permite recuperar y llamar a un método mediante su nombre de cadena. Este enfoque le brinda a su agente la flexibilidad de seleccionar el método de DataFrame más apropiado según las necesidades de análisis actuales.

Al proporcionar tanto el nombre del archivo como el nombre del método como parámetros, el LLM puede elegir de forma inteligente qué técnicas de análisis aplicar a diferentes conjuntos de datos.

El resultado de cada llamada al método se convierte a una representación de cadena, lo que permite al LLM acceder a él para su posterior análisis y razonamiento.

In [7]:
@tool
def llamar_metodo_dataframe(nombre_archivo: str, metodo: str) -> str:

    """
    Ejecuta un método en un DataFrame y devuelve el resultado.
    Esta herramienta te permite ejecutar métodos sencillos de DataFrame, como `head`, `tail` o `describe`,
    en un conjunto de datos que ya se ha cargado y almacenado en caché con `precarga_datasets`.

    Argumentos:
        nombre_archivo (str): La ruta o el nombre del conjunto de datos en la caché global.
        metodo (str): El nombre del método que se va a llamar en el DataFrame. Solo se admiten métodos sin argumentos (p. ej., `head`, `describe`, `info`).

    Devuelve:
        str: La salida del método como una cadena formateada o un mensaje de error si no se encuentra el conjunto de datos o el método no es válido.

    Ejemplo:
        llamar_metodo_dataframe(nombre_archivo= "data.csv", metodo= "head")
    """

    # Intenta obtener el DataFrame de la caché, o cárgalo si aún no está en caché.
    if nombre_archivo not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[nombre_archivo] = pd.read_csv(nombre_archivo)
        except FileNotFoundError:
            return f"No se encontró el DataFrame '{nombre_archivo}' en la caché ni en el disco."
        except Exception as e:
            return f"Error al cargar '{nombre_archivo}': {str(e)}"

    df = DATAFRAME_CACHE[nombre_archivo]
    func = getattr(df, metodo, None)

    if not callable(func):
        return f"'{metodo}' no es un método válido de DataFrame."
    try:
        result = func()

        return str(result)
    except Exception as e:
        return f"Error llamada al '{metodo}' en '{nombre_archivo}': {str(e)}"

### *3.4. Herramientas de Evaluación de Modelos*

Estas herramientas proporcionan funcionalidades especializadas para la creación y evaluación de modelos de aprendizaje automático en conjuntos de datos. El agente primero analizará la estructura del conjunto de datos utilizando herramientas previas para determinar si la tarea de predicción es de clasificación o regresión. Luego, en función de su evaluación, llamará a `evaluate_classification_dataset` o `evaluate_regression_dataset`, proporcionando el nombre del archivo del conjunto de datos y la columna objetivo correspondientes. Ambas herramientas gestionan los aspectos técnicos del aprendizaje automático (división de los datos, entrenamiento del modelo y cálculo de métricas de rendimiento) abstraiendo los detalles de implementación. Para tareas de clasificación, el agente examinará el tipo de datos y los valores únicos de la columna objetivo para determinar si es categórica, y luego llamará al evaluador de clasificación, que devuelve métricas de precisión. Para tareas de regresión que implican predicciones numéricas continuas, el agente seleccionará el evaluador de regresión que devuelve el coeficiente de determinación (R²) y el error cuadrático medio. Este proceso de toma de decisiones muestra cómo un agente bien diseñado puede elegir la herramienta adecuada en función de las características de los datos, demostrando la automatización inteligente del flujo de trabajo.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error

# Se asume que esta caché global es compartida.
# DATAFRAME_CACHE = {}

@tool
def evaluacion_clasificacion_dataset(nombre_archivo: str, columna_objetivo: str) -> Dict[str, float]:
    """
    Entrena y evalúa un clasificador en un conjunto de datos utilizando la columna objetivo especificada.
    Argumentos:
        nombre_archivo (str): El nombre o la ruta del conjunto de datos almacenado en DATAFRAME_CACHE.
        columna_objetivo (str): El nombre de la columna que se utilizará como objetivo de clasificación.
    Devuelve:
        Dict[str, float]: Un diccionario con la puntuación de precisión del modelo.

    """

    # Intenta obtener el DataFrame de la caché, o cárgalo si aún no está en caché.
    if nombre_archivo not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[nombre_archivo] = pd.read_csv(nombre_archivo)
        except FileNotFoundError:
            return {"error": f"DataFrame '{nombre_archivo}' no fue encontrado en el cache o disco."}
        except Exception as e:
            return {"error": f"Error al cargar '{nombre_archivo}': {str(e)}"}
    
    df = DATAFRAME_CACHE[nombre_archivo]

    if columna_objetivo not in df.columns:
        return {"error": f"Columna objetivo '{columna_objetivo}' no fue encontrado en '{columna_objetivo}'."}
    
    X = df.drop(columns= [columna_objetivo])
    y = df[columna_objetivo]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

    model = RandomForestClassifier()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    return {"accuracy": acc}

@tool
def evaluacion_regresion_dataset(nombre_archivo: str, columna_objetivo: str) -> Dict[str, float]:
    """
    Entrena y evalúa un modelo de regresión en un conjunto de datos utilizando la columna objetivo especificada.
    Argumentos:
        nombre_archivo (str): El nombre o la ruta del conjunto de datos almacenado en DATAFRAME_CACHE.
        columna_objetivo (str): El nombre de la columna que se utilizará como objetivo de la regresión.
    Devuelve:
        Dict[str, float]: Un diccionario con el coeficiente de determinación (R²) y el error cuadrático medio (ECM).
    """

    # Intenta obtener el DataFrame de la caché, o cárgalo si aún no está en caché.
    if nombre_archivo not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[nombre_archivo] = pd.read_csv(nombre_archivo)
        except FileNotFoundError:
            return {"error": f"DataFrame '{nombre_archivo}' no fue encontrado en el cache o disco."}
        except Exception as e:
            return {"error": f"Error al cargar '{nombre_archivo}': {str(e)}"}
    
    df = DATAFRAME_CACHE[nombre_archivo]
    
    if columna_objetivo not in df.columns:
        return {"error": f"Columna objetivo '{columna_objetivo}' no fue encontrado en '{columna_objetivo}'."}
    
    X = df.drop(columns= [columna_objetivo])
    y = df[columna_objetivo]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    return {
        "r2_score": r2,
        "mean_squared_error": mse
    }

## 4. Agentes
En LangChain, los agentes son componentes avanzados que permiten a los modelos de IA decidir cuándo y cómo usar las herramientas de forma dinámica. En lugar de depender de scripts predefinidos, los agentes analizan las consultas del usuario y eligen las mejores herramientas para lograr un objetivo. El siguiente paso es definir el agente, lo que requiere especificar cómo debe pensar y comportarse. Se utiliza `ChatPromptTemplate.from_messages()` para crear una solicitud estructurada con tres componentes esenciales:

1. **Mensaje del sistema**: Este establece la identidad y el objetivo principal del agente. Se define como un asistente de ciencia de datos cuya tarea es analizar archivos CSV y determinar si cada conjunto de datos es adecuado para clasificación o regresión según su estructura. Esto le da al agente un propósito y un alcance claros.

2. **Entrada del usuario**: El marcador de posición `{input}` se reemplazará con la consulta real del usuario. Esto permite que el agente responda directamente a la pregunta del usuario.

3. **Bloc de notas del agente**: El marcador de posición `{agent_scratchpad}` es crucial para los agentes que llaman a herramientas, ya que les proporciona espacio para mostrar su proceso de razonamiento y registrar los pasos intermedios. Esto permite al agente construir una cadena de pensamiento, llamar a las herramientas secuencialmente y usar los resultados de una herramienta para tomar decisiones sobre llamadas posteriores.

![imagen de los agentes.png](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/TYkDvBmpmmSXx6TftNpJgw/agents%20copy.png)

[Artículo de referencia para la imagen](https://medium.com/@Shamimw/understanding-langchain-tools-and-agents-a-guide-to-building-smart-ai-applications-e81d200b3c12)


In [9]:
from langchain_classic.agents import create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 🧠 Paso 1: Prompt.

prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "Eres asistente de ciencia de datos. Utiliza las herramientas disponibles para analizar archivos CSV. "
     "Tu trabajo consiste en determinar, en función de su estructura, si cada conjunto de datos es para clasificación o regresión."),
    
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}")  # Requerido para los agentes que realizan llamadas a herramientas.
])

Ahora, crea un objeto de chatbot

In [10]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider= "openai", streaming= False)

Crea una herramienta de lista que contenga todos los objetos de la herramienta

In [11]:
tools = [listar_archivos_csv, precarga_datasets, get_resumenes_dataset, llamar_metodo_dataframe, evaluacion_clasificacion_dataset, evaluacion_regresion_dataset]

### *4.1. Creación y limitaciones del agente*

Aquí, crearás tu agente usando `create_openai_tools_agent()`, que combina el modelo de lenguaje, las herramientas y la plantilla de indicaciones en un agente funcional. Sin embargo, este agente básico tiene limitaciones importantes cuando se usa directamente. Solo realiza un paso de razonamiento y uso de herramientas por invocación, y luego devuelve su proceso de pensamiento intermedio en lugar de una respuesta final. Este comportamiento se debe a que el agente no gestiona automáticamente el ciclo completo de pensar, actuar, observar resultados y continuar razonando hasta alcanzar una solución completa.

In [12]:
# Construir el agente de llamada de la herramienta.
agente = create_openai_tools_agent(llm, tools, prompt)

In [13]:
response = agente.invoke({
    "input": "¿Podrías darme información sobre el conjunto de datos?",
    "intermediate_steps": []
})

In [14]:
# Obtener la primera ToolAgentAction de la lista.
accion = response[0]

# Print the key details
print("🧠 El agente decidió llamar a una herramienta:")
print("Nombre herramienta:", accion.tool)
print("Nombre de inputs de la herramienta:", accion.tool_input)
print("Log:\n", accion.log.strip())

🧠 El agente decidió llamar a una herramienta:
Nombre herramienta: listar_archivos_csv
Nombre de inputs de la herramienta: {}
Log:
 Invoking: `listar_archivos_csv` with `{}`


Cuando se le preguntó al agente "¿Puedes informarme sobre el conjunto de datos?", respondió con una acción de herramienta: optó por invocar la herramienta `list_csv_files` sin argumentos. No intentó cargar ni analizar el conjunto de datos.

Los agentes de estilo ReAct siguen un ciclo de razonamiento paso a paso. ReAct significa Razonamiento y Acción: el agente piensa en qué hacer a continuación, realiza una acción (como invocar una herramienta) y espera el resultado antes de continuar. Por eso, su primer instinto es obtener contexto —listando los archivos CSV disponibles— antes de intentar algo más complejo. Esto no es un fallo; es la forma en que el agente está diseñado para funcionar: razonando paso a paso en función de la retroalimentación.

### *4.2. Ejecutor del agente ReAct*

Gestionar manualmente este bucle ReAct puede resultar engorroso, por lo que utilizará el AgentExecutor. El AgentExecutor encapsula el agente y el conjunto de herramientas, y gestiona todo el ciclo de uso de herramientas de forma transparente. Ejecuta automáticamente el agente, ejecuta la herramienta seleccionada, obtiene el resultado (observación) y lo devuelve al agente hasta que se obtiene una respuesta final. Sin el ejecutor, tendría que gestionar manualmente cada paso, incluyendo la comprobación de si el agente devolvió una llamada a la herramienta o una respuesta final, la ejecución de la herramienta y el seguimiento de los pasos intermedios; todo esto lo gestiona el ejecutor.

In [15]:
from langchain_classic.agents import AgentExecutor

#### 4.2.1. Configuración del ejecutor del agente

La línea `AgentExecutor` crea un sistema de agente completo y autónomo al añadir funcionalidades adicionales a su agente básico. Este ejecutor gestiona el ciclo completo de ReAct (razonamiento y acción), lo que permite al agente realizar múltiples llamadas a herramientas en secuencia hasta llegar a una respuesta final. Configúrelo con los siguientes parámetros importantes:

1. **agent**: El agente que se ejecutará para crear un plan y determinar las acciones a realizar en cada paso del ciclo de ejecución.

2. **tools**: Las herramientas válidas que el agente puede utilizar.

3. **verbose=True**: Permite el registro detallado de cada paso del proceso de razonamiento y llamada a herramientas del agente, lo cual es fundamental para la depuración y la comprensión de cómo el agente llega a sus conclusiones.

4. **handle_parsing_errors=True**: En lugar de fallar, el ejecutor intentará recuperarse y continuar la conversación.

La segunda línea, `agent_executor.agent.stream_runnable = False`, desactiva el modo de transmisión para el agente.

In [16]:
agente_ejecutor = AgentExecutor(agent= agente, tools= tools, verbose= True, handle_parsing_errors= True)
agente_ejecutor.agent.stream_runnable = False

Ahora puedes crear un bot con DataWizard:

In [17]:
print("📊 Haz preguntas sobre tu conjunto de datos (escribe 'exit' para salir):")

while True:
    user_input= input(" You:")
    if user_input.strip().lower() in ['exit', 'quit']:
        print("nos vemos más tarde")
        break
        
    resultado = agente_ejecutor.invoke({"input": user_input})

    print(f"Mi agente: {resultado['output']}")

📊 Haz preguntas sobre tu conjunto de datos (escribe 'exit' para salir):


> Entering new AgentExecutor chain...

Invoking: `listar_archivos_csv` with `{}`


['data/classification-dataset.csv', 'data/regression-dataset.csv']
Invoking: `precarga_datasets` with `{'rutas': ['data/classification-dataset.csv']}`


Cargados datasets: ['data/classification-dataset.csv']
Ya almacenados en cache: []
Invoking: `get_resumenes_dataset` with `{'rutas_dataset': ['data/classification-dataset.csv']}`


[{'nombre_archivo': 'data/classification-dataset.csv', 'nombres_columnas': ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', '

Copyright © IBM Corporation. All rights reserved.
